In [ ]:
import time
import numpy as np
import pandas as pd
import joblib
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import cudf
import cupy as cp
import cuml
import xgboost as xgb

from cuml.pipeline import Pipeline
from cuml.preprocessing import Normalizer
from cuml.decomposition import PCA
from cuml.linear_model import LogisticRegression
from cuml.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
data_path = "../data/spectra/"

data = np.load(data_path + "data.npy")
labels = np.load(data_path + "labels.npy")
wavelengths = np.load(data_path + "wavelengths.npy")

class_names = ["AGN", "galaxy", "QSO"]

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)
print("Wavelength bins:", wavelengths.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data,
    labels.astype("int32"),
    test_size=0.1,
    stratify=labels,
    random_state=42,
    shuffle=True
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:
X_train_gdf = cudf.DataFrame(
    X_train.astype(np.float32)
)

y_train_gdf = cudf.Series(
    y_train.astype(np.int32)
)

X_test_gdf = cudf.DataFrame(
    X_test.astype(np.float32)
)

y_test_gdf = cudf.Series(
    y_test.astype(np.int32)
)

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
)

In [ ]:
def objective(trial, X, y, cv, model_type):

    # ---------------------------------------------------------
    # PCA configuration
    # ---------------------------------------------------------

    use_pca = trial.suggest_categorical(
        "use_pca",
        [True, False]
    )

    if use_pca:

        # cuML PCA does not reliably support sklearn-style
        # explained variance ratios such as 0.95
        # therefore we optimize integer component counts

        pca_n = trial.suggest_int(
            "pca_n_components",
            5,
            100
        )

        pca = PCA(
            n_components=pca_n,
        )

    else:
        pca = "passthrough"

    # ---------------------------------------------------------
    # Model selection
    # ---------------------------------------------------------

    if model_type == "logit":

        clf = LogisticRegression(
            C=trial.suggest_float(
                "logit_C",
                1e-1,
                1e4,
                log=True
            ),
            max_iter=10000,
            solver="qn"
        )

    elif model_type == "rf":

        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int(
                "rf_n_estimators",
                100,
                400
            ),

            max_depth=trial.suggest_categorical(
                "rf_max_depth",
                [5, 10, 20, None]
            ),

            max_features=trial.suggest_categorical(
                "rf_max_features",
                ["sqrt", "log2"]
            ),

            min_samples_leaf=trial.suggest_categorical(
                "rf_min_samples_leaf",
                [1, 2, 5]
            ),

        )

    elif model_type == "xgb":

        clf = xgb.XGBClassifier(

            objective="multi:softprob",
            eval_metric="mlogloss",

            tree_method="hist",
            device="cuda",

            n_estimators=trial.suggest_int(
                "xgb_n_estimators",
                100,
                500
            ),

            learning_rate=trial.suggest_float(
                "xgb_learning_rate",
                1e-3,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_categorical(
                "xgb_max_depth",
                [3, 4, 5, 6]
            ),

            subsample=trial.suggest_categorical(
                "xgb_subsample",
                [0.6, 0.8, 1.0]
            ),

            colsample_bytree=trial.suggest_categorical(
                "xgb_colsample_bytree",
                [0.5, 0.8, 1.0]
            ),

            min_child_weight=trial.suggest_categorical(
                "xgb_min_child_weight",
                [1, 3, 5]
            )
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    # ---------------------------------------------------------
    # Pipeline
    # ---------------------------------------------------------

    steps = [
        ("normalize", Normalizer(norm="l2"))
    ]

    if pca != "passthrough":
        steps.append(("pca", pca))

    steps.append(("classifier", clf))

    pipe = Pipeline(steps)

    # ---------------------------------------------------------
    # Cross-validation
    # ---------------------------------------------------------

    scores = []

    # sklearn splitter operates on CPU arrays
    for train_idx, val_idx in cv.split(
        X.to_pandas(),
        y.to_pandas()
    ):

        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # fit
        pipe.fit(
            X_train_fold,
            y_train_fold
        )

        # predict
        preds = pipe.predict(X_val_fold)

        # move to CPU for sklearn metrics
        # Convert ground truth safely
        y_true = y_val_fold.to_numpy()
        
        # Convert predictions safely
        if hasattr(preds, "to_numpy"):
            y_pred = preds.to_numpy()
        
        elif isinstance(preds, cp.ndarray):
            y_pred = cp.asnumpy(preds)
        
        else:
            y_pred = np.asarray(preds)
        
        score = f1_score(
            y_true,
            y_pred,
            average="macro"
        )

        scores.append(score)

    return float(np.mean(scores))

In [ ]:
all_results = {}

best_overall_score = -np.inf
best_overall_model = None
best_model_name = None

for model_name in ["logit", "rf", "xgb"]:

    print("\n" + "=" * 60)
    print(f"Optimizing {model_name}")
    print("=" * 60)

    study = optuna.create_study(
        direction="maximize"
    )

    start = time.time()

    study.optimize(
        lambda trial: objective(
            trial,
            X_train_gdf,
            y_train_gdf,
            cv,
            model_name
        ),
        n_trials=20
    )

    elapsed = time.time() - start

    print("\nBest CV F1:")
    print(f"{study.best_value:.4f}")

    print("\nBest Parameters:")
    print(study.best_params)

    all_results[model_name] = {
        "best_score": study.best_value,
        "best_params": study.best_params,
        "fit_time_sec": elapsed
    }

    # track best overall model
    if study.best_value > best_overall_score:

        best_overall_score = study.best_value
        best_model_name = model_name

In [ ]:
best_params = all_results[best_model_name]["best_params"]

print("\nBest overall model:")
print(best_model_name)

steps = [
    ("normalize", Normalizer(norm="l2"))
]

# PCA
if best_params.get("use_pca", False):

    steps.append((
        "pca",
        PCA(
            n_components=best_params["pca_n_components"],
        )
    ))

# Classifier
if best_model_name == "logit":

    clf = LogisticRegression(
        C=best_params["logit_C"],
        max_iter=10000,
        solver="qn"
    )

elif best_model_name == "rf":

    clf = RandomForestClassifier(
        n_estimators=best_params["rf_n_estimators"],
        max_depth=best_params["rf_max_depth"],
        max_features=best_params["rf_max_features"],
        min_samples_leaf=best_params["rf_min_samples_leaf"],
    )

elif best_model_name == "xgb":

    clf = xgb.XGBClassifier(

        objective="multi:softprob",
        eval_metric="mlogloss",

        tree_method="hist",
        device="cuda",


        n_estimators=best_params["xgb_n_estimators"],
        learning_rate=best_params["xgb_learning_rate"],
        max_depth=best_params["xgb_max_depth"],
        subsample=best_params["xgb_subsample"],
        colsample_bytree=best_params["xgb_colsample_bytree"],
        min_child_weight=best_params["xgb_min_child_weight"]
    )

steps.append(("classifier", clf))

final_model = Pipeline(steps)

In [ ]:
final_model.fit(
    X_train_gdf,
    y_train_gdf
)

In [ ]:
test_predictions = final_model.predict(
    X_test_gdf
)

# move back to CPU
y_test_cpu = y_test_gdf.to_numpy()
pred_cpu = test_predictions

test_accuracy = accuracy_score(
    y_test_cpu,
    pred_cpu
)

test_f1 = f1_score(
    y_test_cpu,
    pred_cpu,
    average="macro"
)

print(f"Final Test Accuracy: {test_accuracy:.4f}")
print(f"Final Test F1 Macro: {test_f1:.4f}")

In [ ]:
print(
    classification_report(
        y_test_cpu,
        pred_cpu,
        target_names=class_names
    )
)

In [ ]:
def get_multiclass_metrics(y_actual, y_pred, classes):
    cm = confusion_matrix(y_actual, y_pred)
    
    for i, cls in enumerate(classes):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - TP - FP - FN

        # Safe division to prevent ZeroDivisionError
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

        print(f"{cls:10s} Sensitivity: {sensitivity:.4f} | Specificity: {specificity:.4f}")

def report(y_actual, y_pred, classes, title):
    print(classification_report(y_actual, y_pred, target_names=classes))
    print("\nPer-class metrics:")
    get_multiclass_metrics(y_actual, y_pred, classes)
    cm = confusion_matrix(y_actual, y_pred, normalize="true")

    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="viridis", 
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

In [ ]:
report(y_test, test_predictions, class_names,
       title="Final Test Confusion Matrix")


In [ ]:
joblib.dump(
    final_model,
    "best_spectrum_classifier_gpu_optuna.pkl"
)